# Notebook for merging and cleaning deidentified "book a librarian" export data

In [1]:
import pandas as pd
import numpy as np
import plotly.express as px
import openpyxl

In [2]:
# Book a Librarian files
BOOKING_FILES = [
    "Bookings Report Data AY21-22 - Deidentified.xlsx",
    "Book A Librarian Export Q1 AY22-23 - Deidentified.xlsx",
    "Book A Librarian Export Q2 AY22-23 - Deidentified.xlsx",
    "Book A Librarian Export Q3 AY22-23 - Deidentified.xlsx",
    "Book A Librarian Export Q4 AY22-23 - Deidentified.xlsx",
    "Book a Librarian Export Q1 AY23-24 - Deidentified.xlsx",
    "Book a Librarian Export Q2 AY23-24 - Deidentified.xlsx",
    "Book a Librarian Export Q3 AY23-24 - Deidentified.xlsx",
    "Book a Librarian Export Q4 AY23-24 - Deidentified.xlsx",
    "Book a Librarian Export Q1 AY24-25 - Deidentified.xlsx",
    "Book a Librarian Export Q2 AY24-25 - Deidentified.xlsx",
    "Book a Librarian Export Q3 AY24-25 - Deidentified.xlsx",
    "Book a Librarian Export Q4 AY24-25 - Deidentified.xlsx",
]

In [3]:
frames = []
for fname in BOOKING_FILES:
    fpath = '/home/ekmys/PycharmProjects/Your-Library-Your-Impact/dashboard/data/' + fname
    df = pd.read_excel(fpath)
    frames.append(df)

In [4]:
combined = pd.concat(frames, ignore_index=True)

In [5]:
combined.head()

,Date,Service,Location,Duration (mins.),Signed Up Attendees Count,Status,Program/Dept,Topic,Event Type,Booking Id,Tracking Data,PNWU Status,PNWU Academic Affiliation,Additional Notes,PNWU Status
0,2022-04-04,Research Consultation,Virtual,60,1,Student,DO,"I'm available any time next week, but I'd like...",Single,AAMkAGM1ZDA4MDNjLTUyOWUtNGVlNi04ZmZmLTU5NTU5OD...,NaN,NaN,NaN,NaN,NaN
1,2022-04-08,Library Access Orientation,Virtual,30,1,University,Staff,null,Single,AAMkAGM1ZDA4MDNjLTUyOWUtNGVlNi04ZmZmLTU5NTU5OD...,NaN,NaN,NaN,NaN,NaN
2,2022-04-22,Research Consultation,Virtual,60,1,Student,DO,Question about how to get publish and where an...,Single,AAMkAGM1ZDA4MDNjLTUyOWUtNGVlNi04ZmZmLTU5NTU5OD...,NaN,NaN,NaN,NaN,NaN
3,2022-05-05,Research Consultation,Virtual,60,1,Fulltime Faculty,Staff,I need help searching for a journal for an art...,Single,AAMkAGM1ZDA4MDNjLTUyOWUtNGVlNi04ZmZmLTU5NTU5OD...,NaN,NaN,NaN,NaN,NaN
4,2022-05-09,Special Project/Collaboration,PNWU Library or Virtual,60,1,University,Staff,NaN,Single,AAMkAGM1ZDA4MDNjLTUyOWUtNGVlNi04ZmZmLTU5NTU5OD...,NaN,NaN,NaN,NaN,NaN


In [6]:
combined.info()

<class 'pandas.DataFrame'>
RangeIndex: 454 entries, 0 to 453
Data columns (total 15 columns):
 #   Column                     Non-Null Count  Dtype         
---  ------                     --------------  -----         
 0   Date                       454 non-null    datetime64[us]
 1   Service                    454 non-null    str           
 2   Location                   446 non-null    str           
 3   Duration (mins.)           454 non-null    int64         
 4   Signed Up Attendees Count  454 non-null    int64         
 5   Status                     12 non-null     str           
 6   Program/Dept               12 non-null     str           
 7   Topic                      448 non-null    str           
 8   Event Type                 12 non-null     str           
 9   Booking Id                 12 non-null     str           
 10  Tracking Data              0 non-null      float64       
 11  PNWU Status                410 non-null    str           
 12  PNWU Academic Affil

In [7]:
# Parse dates so we can sort and group by time
combined["Date"] = pd.to_datetime(combined["Date"], errors="coerce")
combined = combined.dropna(subset=["Date"])

In [8]:
# Add a Year-Quarter label for grouping in charts
combined["YearQuarter"] = (
    combined["Date"].dt.year.astype(str)
    + " Q"
    + combined["Date"].dt.quarter.astype(str)
)

In [9]:
# Add academic year label (AY runs July-June)
# July 2022 - June 2023 = AY22-23
def get_academic_year(date):
    if date.month >= 7:
        return f"AY{str(date.year)[2:]}-{str(date.year+1)[2:]}"
    else:
        return f"AY{str(date.year-1)[2:]}-{str(date.year)[2:]}"

In [11]:
combined["AcademicYear"] = combined["Date"].apply(get_academic_year)
# Some files had extra whitespaces or diff caps
# or making dupes, this  should clean it up
combined["Service"] = combined["Service"].str.strip()
combined["Location"] = combined["Location"].str.strip()
combined["Topic"] = combined["Topic"].str.strip()
combined["PNWU Status"] = combined["PNWU Status"].str.strip()
combined["PNWU Academic Affiliation"] = combined["PNWU Academic Affiliation"].str.strip()

In [27]:
# for bioethics topics - consolidate multiple-category topics for better visualization
combined['Topic'] = combined['Topic'].replace(
    'Physician Aid-in-Dying AND Do Not Resuscitate Orders AND End-of-Life Issues AND Parental Decision Making',
    'Physician Aid in Dying')
combined['Topic'] = combined['Topic'].replace(
    'End-of-Life Issues AND (Treatment Refusal OR Cross-Cultural Issues and Diverse Beliefs)', 'End-of-Life Issues')
combined['Topic'] = combined['Topic'].replace('End-of-Life Issues OR Termination of Life-Sustaining Treatment', 'End-of-Life Issues')
combined['Topic'] = combined['Topic'].replace('Public Health Ethics AND Mistakes', 'Public Health Ethics')

In [12]:
# drop unnecessary columns
combined = combined.drop(['Status', 'Program/Dept', 'Booking Id','Tracking Data', ' PNWU Status'], axis=1)

In [13]:
combined.head()

,Date,Service,Location,Duration (mins.),Signed Up Attendees Count,Topic,Event Type,PNWU Status,PNWU Academic Affiliation,Additional Notes,YearQuarter,AcademicYear
0,2022-04-04,Research Consultation,Virtual,60,1,"I'm available any time next week, but I'd like...",Single,NaN,NaN,NaN,2022 Q2,AY21-22
1,2022-04-08,Library Access Orientation,Virtual,30,1,null,Single,NaN,NaN,NaN,2022 Q2,AY21-22
2,2022-04-22,Research Consultation,Virtual,60,1,Question about how to get publish and where an...,Single,NaN,NaN,NaN,2022 Q2,AY21-22
3,2022-05-05,Research Consultation,Virtual,60,1,I need help searching for a journal for an art...,Single,NaN,NaN,NaN,2022 Q2,AY21-22
4,2022-05-09,Special Project/Collaboration,PNWU Library or Virtual,60,1,NaN,Single,NaN,NaN,NaN,2022 Q2,AY21-22


In [14]:
display(combined)

,Date,Service,Location,Duration (mins.),Signed Up Attendees Count,Topic,Event Type,PNWU Status,PNWU Academic Affiliation,Additional Notes,YearQuarter,AcademicYear
0,2022-04-04,Research Consultation,Virtual,60,1,"I'm available any time next week, but I'd like...",Single,NaN,NaN,NaN,2022 Q2,AY21-22
1,2022-04-08,Library Access Orientation,Virtual,30,1,null,Single,NaN,NaN,NaN,2022 Q2,AY21-22
2,2022-04-22,Research Consultation,Virtual,60,1,Question about how to get publish and where an...,Single,NaN,NaN,NaN,2022 Q2,AY21-22
3,2022-05-05,Research Consultation,Virtual,60,1,I need help searching for a journal for an art...,Single,NaN,NaN,NaN,2022 Q2,AY21-22
4,2022-05-09,Special Project/Collaboration,PNWU Library or Virtual,60,1,NaN,Single,NaN,NaN,NaN,2022 Q2,AY21-22
...,...,...,...,...,...,...,...,...,...,...,...,...
449,2025-06-17,Research Consultation,Virtual,60,1,Tell us about your research project in 3 sente...,NaN,Student,DO,NaN,2025 Q2,AY24-25
450,2025-05-05,Special Project/Collaboration,Virtual,60,1,What would you like to discuss?: OregonTech Li...,NaN,Affiliate,Not Applicable,NaN,2025 Q2,AY24-25
451,2025-05-07,Special Project/Collaboration,Virtual,60,1,What would you like to discuss?: Vendor-Contra...,NaN,Staff,Not Applicable,NaN,2025 Q2,AY24-25
452,2025-04-02,Special Project/Collaboration,Virtual,60,1,What would you like to discuss?: Transition fr...,NaN,Faculty,DO,NaN,2025 Q2,AY24-25


# Making sure charts still work

In [28]:
df = combined

# PNWU Brand Colors
PACIFIC_BLUE    = "#1b3764"
FOREST_GREEN    = "#366732"
VINEYARD_GREEN  = "#659a41"
NEW_LEAF        = "#99ca3c"
CLOUD_BLUE      = "#72c7f0"
SILVER_GRAY     = "#a4a9ad"
BALANCE_GRAY    = "#616467"
WARNING_RED     = "#c0392b"

In [17]:
# making sure charts function still works

# plot bookings by year
agg = (
    df.groupby("AcademicYear", as_index=False)
    .size()
    .rename(columns={"size": "Appointments"})
    .sort_values("AcademicYear")
)

fig = px.bar(
    agg,
    x="AcademicYear",
    y="Appointments",
    text="Appointments",
    color_discrete_sequence=[CLOUD_BLUE],
)
fig.update_traces(textposition="outside")
fig.update_layout(
    title="Book a Librarian — Appointments by Academic Year",
    xaxis_title="Academic Year",
    yaxis_title="Total Appointments",
    plot_bgcolor="white",
    paper_bgcolor="white",
    yaxis=dict(gridcolor="#f0f0f0"),
)

fig.show()

In [18]:
# bookings by quarter

import streamlit as st

agg = (
    df.groupby("YearQuarter", as_index=False)
    .size()
    .rename(columns={"size": "Appointments"})
    .sort_values("YearQuarter")
)

st.area_chart(
    agg.set_index("YearQuarter")["Appointments"],
    color=FOREST_GREEN,
)

2026-05-22 16:20:28.720 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-05-22 16:20:28.838 
  command:

    streamlit run /home/ekmys/PycharmProjects/Your-Library-Your-Impact/.venv/lib/python3.14/site-packages/ipykernel_launcher.py [ARGUMENTS]
2026-05-22 16:20:28.839 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-05-22 16:20:28.840 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.


DeltaGenerator()

In [19]:
# plot service type

agg = (
    df["Service"]
    .value_counts()
    .reset_index()
    .rename(columns={"index": "Service", "count": "Count"})
)

fig = px.bar(
    agg,
    x="Count",
    y="Service",
    orientation="h",
    text="Count",
    color_discrete_sequence=[VINEYARD_GREEN],
)
fig.update_traces(textposition="outside")
fig.update_layout(
    title="Appointments by Service Type",
    xaxis_title="Number of Appointments",
    yaxis_title="",
    plot_bgcolor="white",
    paper_bgcolor="white",
    xaxis=dict(gridcolor="#f0f0f0"),
)

fig.show()

In [21]:
# plot virtual vs. in person

# Renamed for clarity
location_map = {
    "Virtual": "Virtual",
    "Library": "In-Person",
    "PNWU Library or Virtual": "In-Person",
}
df = df.copy()
df["Location"] = df["Location"].replace(location_map)

agg = df["Location"].value_counts().reset_index()
agg.columns = ["Location", "Count"]

fig = px.pie(
    agg,
    names="Location",
    values="Count",
    color_discrete_sequence=[PACIFIC_BLUE, CLOUD_BLUE],
    hole=0.4,
)
fig.update_layout(title="Appointment Delivery Method")

fig.show()

# Bioethics chart

In [ ]:
def plot_bioethics_topics(df: pd.DataFrame) -> None:
    """Plots bioethics topics discussed in Book A Librarian appointments.
    """
    if df.empty:
        st.info("No data available.")
        return

    # create bioethics only dataframe
    df_bioethics = df[df['Service'] == 'ELEC 704 Bioethics Consultation']

    # create groupby object
    agg = (df_bioethics.groupby("Topic", as_index=False).size())

    # PLOTLY BARPLOT VERSION
    labels = {'Topic':'Bioethics Topic', 'size':'Number of Bookings'}
    fig = px.bar(agg, x=agg['Topic'], y=agg['size'], labels=labels, title='Number of Bookings by Bioethics Topic')
    fig.update_xaxes(tickangle=45)
    st.plotly_chart(fig, width='stretch')

In [29]:
bioethics_df = df[df['Service'] == 'ELEC 704 Bioethics Consultation']

In [30]:
bioethics_df.head()

,Date,Service,Location,Duration (mins.),Signed Up Attendees Count,Topic,Event Type,PNWU Status,PNWU Academic Affiliation,Additional Notes,YearQuarter,AcademicYear
248,2024-01-16,ELEC 704 Bioethics Consultation,Virtual,60,1,Parental Decision Making,NaN,Student,NaN,NaN,2024 Q1,AY23-24
249,2024-01-17,ELEC 704 Bioethics Consultation,Virtual,60,1,Ethics Committees and Consultation,NaN,Student,NaN,NaN,2024 Q1,AY23-24
250,2024-02-02,ELEC 704 Bioethics Consultation,Virtual,60,1,Clinical Ethics and Law,NaN,Student,NaN,NaN,2024 Q1,AY23-24
251,2024-02-05,ELEC 704 Bioethics Consultation,Virtual,60,1,Unsure,NaN,Student,NaN,NaN,2024 Q1,AY23-24
252,2024-02-07,ELEC 704 Bioethics Consultation,Virtual,60,1,Physician Aid-in-Dying,NaN,Student,NaN,NaN,2024 Q1,AY23-24


In [24]:
bioethics_df.info()

<class 'pandas.DataFrame'>
Index: 50 entries, 248 to 433
Data columns (total 12 columns):
 #   Column                     Non-Null Count  Dtype         
---  ------                     --------------  -----         
 0   Date                       50 non-null     datetime64[us]
 1   Service                    50 non-null     str           
 2   Location                   47 non-null     str           
 3   Duration (mins.)           50 non-null     int64         
 4   Signed Up Attendees Count  50 non-null     int64         
 5   Topic                      50 non-null     str           
 6   Event Type                 0 non-null      str           
 7   PNWU Status                45 non-null     str           
 8   PNWU Academic Affiliation  0 non-null      str           
 9   Additional Notes           2 non-null      object        
 10  YearQuarter                50 non-null     str           
 11  AcademicYear               50 non-null     str           
dtypes: datetime64[us](1), i

In [25]:
# create groupby object
agg = (bioethics_df.groupby("Topic", as_index=False).size())

In [26]:
# PLOTLY BARPLOT VERSION
labels = {'Topic':'Bioethics Topic', 'size':'Number of Bookings'}
fig = px.bar(agg, x=agg['Topic'], y=agg['size'], labels=labels, title='Number of Bookings by Bioethics Topic')
fig.update_xaxes(tickangle=45)
fig.show()

In [ ]:
bioethics_df.info()

In [ ]:
def load_bookings() -> pd.DataFrame:
    """
    Load and combine all Book a Librarian appointment files.

    Returns a single dataframe with all appointments across all years,
    plus a YearQuarter column (2022 Q1) for easy grouping

    If a file is missing, it's silently skipped, adding new
    quarters later is easy just drop a new file in data/
    and adding its name to BOOKING_FILES
    """
    frames = []
    for fname in BOOKING_FILES:
        fpath = DATA_DIR / fname
        if not fpath.exists():
            continue
        df = pd.read_excel(fpath)
        frames.append(df)

    if not frames:
        return pd.DataFrame()

    combined = pd.concat(frames, ignore_index=True)

    # Parse dates so we can sort and group by time
    combined["Date"] = pd.to_datetime(combined["Date"], errors="coerce")
    combined = combined.dropna(subset=["Date"])

    # Add a Year-Quarter label for grouping in charts
    combined["YearQuarter"] = (
        combined["Date"].dt.year.astype(str)
        + " Q"
        + combined["Date"].dt.quarter.astype(str)
    )

    # Add academic year label (AY runs July-June)
    # July 2022 - June 2023 = AY22-23
    def get_academic_year(date):
        if date.month >= 7:
            return f"AY{str(date.year)[2:]}-{str(date.year+1)[2:]}"
        else:
            return f"AY{str(date.year-1)[2:]}-{str(date.year)[2:]}"

    combined["AcademicYear"] = combined["Date"].apply(get_academic_year)
    # Some files had extra whitespaces or diff caps
    # or making dupes, this  should clean it up
    combined["Service"] = combined["Service"].str.strip()
    combined["Location"] = combined["Location"].str.strip()
    combined["Topic"] = combined["Topic"].str.strip()

    # for bioethics topics - consolidate multiple-category topics for better visualization
    combined['Topic'] = combined['Topic'].replace(
        'Physician Aid-in-Dying AND Do Not Resuscitate Orders AND End-of-Life Issues AND Parental Decision Making',
        'Physician Aid in Dying')
    combined['Topic'] = combined['Topic'].replace(
        'End-of-Life Issues AND (Treatment Refusal OR Cross-Cultural Issues and Diverse Beliefs)', 'End-of-Life Issues')
    combined['Topic'] = combined['Topic'].replace('End-of-Life Issues OR Termination of Life-Sustaining Treatment',
                                                  'End-of-Life Issues')
    combined['Topic'] = combined['Topic'].replace('Public Health Ethics AND Mistakes', 'Public Health Ethics')

    return combined

In [ ]:
def load_bookings_bioethics() -> pd.DataFrame:
    # read in excel file
    df_bioethics = pd.read_excel('/home/ekmys/PycharmProjects/Your-Library-Your-Impact/dashboard/data/book a librarian_bioethics/BAL_bioethics_all.xlsx')

    # Parse dates so we can sort and group by time
    df_bioethics["Date"] = pd.to_datetime(df_bioethics["Date"], errors="coerce")
    df_bioethics = df_bioethics.dropna(subset=["Date"])

    # Add a Year-Quarter label for grouping in charts
    df_bioethics["YearQuarter"] = (
    df_bioethics["Date"].dt.year.astype(str)
    + " Q"
    + df_bioethics["Date"].dt.quarter.astype(str))

    # rename columns
    df_bioethics = df_bioethics.rename(columns={" Custom Fields (Topics)": "Bioethics Topic"})

    # remove whitespace at head or tail
    df_bioethics['Bioethics Topic'] = df_bioethics['Bioethics Topic'].str.strip()

    # consolidate multiple-category topics
    df_bioethics['Bioethics Topic'] = df_bioethics['Bioethics Topic'].replace('Physician Aid-in-Dying AND Do Not Resuscitate Orders AND End-of-Life Issues AND Parental Decision Making','Physician Aid in Dying')
    df_bioethics['Bioethics Topic'] = df_bioethics['Bioethics Topic'].replace('End-of-Life Issues AND (Treatment Refusal OR Cross-Cultural Issues and Diverse Beliefs)', 'End-of-Life Issues')
    df_bioethics['Bioethics Topic'] = df_bioethics['Bioethics Topic'].replace('End-of-Life Issues OR Termination of Life-Sustaining Treatment', 'End-of-Life Issues')
    df_bioethics['Bioethics Topic'] = df_bioethics['Bioethics Topic'].replace('Public Health Ethics AND Mistakes', 'Public Health Ethics')

    return df_bioethics

In [ ]:
## create dataframes for book a librarian datasets

# 2022-2023
df_book2223Q1 = pd.read_excel('/home/ekmys/PycharmProjects/Your-Library-Your-Impact/data/Book A Librarian Export Q1 AY22-23 - Deidentified.xlsx')
df_book2223Q2 = pd.read_excel('/home/ekmys/PycharmProjects/Your-Library-Your-Impact/data/Book A Librarian Export Q2 AY22-23 - Deidentified.xlsx')
df_book2223Q3 = pd.read_excel('/home/ekmys/PycharmProjects/Your-Library-Your-Impact/data/Book A Librarian Export Q3 AY22-23 - Deidentified.xlsx')
df_book2223Q4 = pd.read_excel('/home/ekmys/PycharmProjects/Your-Library-Your-Impact/data/Book A Librarian Export Q4 AY22-23 - Deidentified.xlsx')

# 2023-2024
df_book2324Q1 = pd.read_excel('/home/ekmys/PycharmProjects/Your-Library-Your-Impact/data/Book a Librarian Export Q1 AY23-24 - Deidentified.xlsx')
df_book2324Q2 = pd.read_excel('/home/ekmys/PycharmProjects/Your-Library-Your-Impact/data/Book a Librarian Export Q2 AY23-24 - Deidentified.xlsx')
df_book2324Q3 = pd.read_excel('/home/ekmys/PycharmProjects/Your-Library-Your-Impact/data/Book a Librarian Export Q3 AY23-24 - Deidentified.xlsx')
df_book2324Q4 = pd.read_excel('/home/ekmys/PycharmProjects/Your-Library-Your-Impact/data/Book a Librarian Export Q4 AY23-24 - Deidentified.xlsx')

# 2024-2025
df_book2425Q1 = pd.read_excel('/home/ekmys/PycharmProjects/Your-Library-Your-Impact/data/Book a Librarian Export Q1 AY24-25 - Deindentified.xlsx')
df_book2425Q2 = pd.read_excel('/home/ekmys/PycharmProjects/Your-Library-Your-Impact/data/Book a Librarian Export Q2 AY24-25 - Deidentified.xlsx')
df_book2425Q3 = pd.read_excel('/home/ekmys/PycharmProjects/Your-Library-Your-Impact/data/Book a Librarian Export Q3 AY24-25 - Deidentified.xlsx')
df_book2425Q4 = pd.read_excel('/home/ekmys/PycharmProjects/Your-Library-Your-Impact/data/Book a Librarian Export Q4 AY24-25 - Deidentified.xlsx')

In [ ]:
# concatenate into one dataframe
frames = [df_book2223Q1, df_book2223Q2, df_book2223Q3, df_book2223Q4,
          df_book2324Q1, df_book2324Q2, df_book2324Q3, df_book2324Q4,
          df_book2425Q1, df_book2425Q2, df_book2425Q3, df_book2425Q4]
df_book_all = pd.concat(frames, axis=0)
df_book_all.head()

In [ ]:
# bookings dataframe info
df_book_all.info()

In [ ]:
# bookings dataframe stats
df_book_all.describe().T

In [ ]:
# drop columns with null values
df_book_all.drop(['Column4', 'Column5', 'Column6', 'Column7'], axis=1, inplace=True)

In [ ]:
df_book_all.head()

# Remove whitespace

In [ ]:
# remove whitespace at head & tail of string-type columns
for column in df_book_all.columns:
    if df_book_all[column].dtype == 'str':
        df_book_all[column] = df_book_all[column].str.strip()

# remove whitespace from any single-word columns
# single_word_cols_VAERS = list(df_VAERS[['RECVDATE','STATE','RPT_DATE','DIED','DATEDIED','L_THREAT','ER_VISIT','HOSPITAL','X_STAY','DISABLE','BIRTH_DEFECT','RECOVD','VAX_DATE','ONSET_DATE','V_ADMINBY','V_FUNDBY']])
# for col in single_word_cols_VAERS:
    # df_VAERS[col] = df_VAERS[col].replace(" ","")

# Unique string values

In [ ]:
# list of unique services
unique_service = list(df_book_all['Service'].unique())
print(unique_service)

In [ ]:
# list of unique locations
unique_location = list(df_book_all['Location'].unique())
print(unique_location)

In [ ]:
# rename columns for "Custom Fields.1, 2 and 3"
df_book_all.rename(columns={"Custom Fields.1": "Custom1", "Custom Fields.2": "Custom2", "Custom Fields.3" : "Custom3"}, inplace=True)

In [ ]:
df_book_all.head()

In [ ]:
# list of unique custom fields
unique_custom1 = list(df_book_all['Custom1'].unique())
# unique_custom2 = list(df_book_all['Custom Fields.2'].unique())
# unique_custom3 = list(df_book_all['Custom Fields.3'].unique())
print(unique_custom1)
# print(unique_custom2)
# print(unique_custom3)

# Visualization planning